# 🚀 MLflow Quickstart for RipCatch

Welcome to the RipCatch MLflow integration! This notebook will guide you through:
- Installing and configuring MLflow
- Setting up experiment tracking
- Logging your first RipCatch experiment
- Viewing results in the MLflow UI

## 📋 Prerequisites
- RipCatch v2.0 model and dataset
- Python 3.8+
- Basic knowledge of YOLOv8 and machine learning

## 📦 Step 1: Install MLflow and Dependencies

In [ ]:
# ============================================================================
# STEP 1: Install MLflow and Dependencies
# ============================================================================

# Install required packages
print("📦 Installing MLflow and dependencies...")
!pip install mlflow psutil matplotlib seaborn plotly ultralytics

# Fix PyTorch CUDA error by reinstalling CPU-only version
print("\n🔧 Fixing PyTorch installation...")
!pip uninstall torch torchvision torchaudio -y
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# Verify installations
import mlflow
print(f"\n✅ MLflow version: {mlflow.__version__}")

import torch
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")

📦 Installing MLflow and dependencies...
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached cryptography-46.0.4-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached fastapi-0.128.0-py3-none-any.whl.metadata (30 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_proto-1.39.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached sqlparse-0.5.5-py3-none-any.whl.metadata (4.7

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-intel 2.12.0 requires numpy<1.24,>=1.22, but you have numpy 1.26.4 which is incompatible.
tensorflow-intel 2.12.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\srava\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



✅ MLflow version: 3.9.0


OSError: [WinError 127] The specified procedure could not be found. Error loading "C:\Users\srava\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torch\lib\c10_cuda.dll" or one of its dependencies.

## 🔧 Step 2: GPU Detection & Hardware Configuration

In [ ]:
# ============================================================================
# STEP 2: GPU Detection & Hardware Configuration
# ============================================================================
# This cell detects your hardware and configures optimal settings
# Based on proven RipCatch-v2.0 setup

import os
import torch
from pathlib import Path

# Set PyTorch CUDA memory allocator (prevents fragmentation)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print("🌊 RIP CURRENT DETECTION - MLFLOW SETUP")
print("=" * 60)

# ==================== GPU DETECTION ====================
print("\n📊 Hardware Detection:")

# Check PyTorch version
pytorch_version = torch.__version__
print(f"  PyTorch: {pytorch_version}")

# Detect CUDA availability
if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cuda_version = torch.version.cuda
    
    print(f"  GPU: {gpu_name}")
    print(f"  VRAM: {gpu_memory_gb:.1f} GB")
    print(f"  CUDA: {cuda_version}")
    
    # Enable cuDNN benchmark for RTX GPUs (faster)
    if "RTX" in gpu_name or "GeForce" in gpu_name:
        torch.backends.cudnn.benchmark = True
        print("  cuDNN Benchmark: Enabled ⚡")
    
    # Set memory fraction to 85% (conservative to prevent OOM)
    torch.cuda.set_per_process_memory_fraction(0.85, 0)
    print("  Memory Limit: 85% (safety margin)")
    
    # Empty cache to start fresh
    torch.cuda.empty_cache()
    
else:
    device = "cpu"
    gpu_name = "CPU"
    gpu_memory_gb = 0
    print("  Mode: CPU (No CUDA detected)")
    print("  ⚠️  Training/validation will be slower on CPU")

# ==================== OPTIMAL SETTINGS ====================
print("\n⚙️  Optimal Configuration:")

# Intelligent batch size selection based on VRAM
if gpu_memory_gb >= 12:  # RTX 3080 Ti, 3090, 4080, etc.
    batch_size = 32
    image_size = 768
elif gpu_memory_gb >= 9.5:  # RTX 3080 (10GB)
    batch_size = 32
    image_size = 768
elif gpu_memory_gb >= 8:  # RTX 3070, 2080, etc.
    batch_size = 28
    image_size = 768
elif gpu_memory_gb >= 6:  # RTX 3060, 2060, etc.
    batch_size = 12
    image_size = 640
else:  # CPU or low VRAM
    batch_size = 8
    image_size = 640

print(f"  Batch Size: {batch_size}")
print(f"  Image Size: {image_size}px")
print(f"  Device: {device}")

# ==================== STATUS SUMMARY ====================
print("\n" + "=" * 60)
if device == "cuda":
    status = "✅ GPU READY FOR TRAINING"
    emoji = "🚀"
else:
    status = "⚠️  CPU MODE (SLOWER PERFORMANCE)"
    emoji = "🐌"

print(f"{emoji} Status: {status}")
print("=" * 60)

# Save config for later use
config = {
    'device': device,
    'gpu_name': gpu_name,
    'gpu_memory_gb': gpu_memory_gb,
    'batch_size': batch_size,
    'image_size': image_size,
}

print("\n💡 TIP: Hardware configuration saved to 'config' variable")
print("   Access with: config['device'], config['batch_size'], etc.")

## 🔧 Step 3: Configure MLflow Experiment Tracking

In [ ]:
# ============================================================================
# STEP 3: Configure MLflow Experiment Tracking
# ============================================================================
# This sets up MLflow to track your experiments

import sys
from pathlib import Path
import mlflow
from mlflow import log_metric, log_param, log_artifacts
from ultralytics import YOLO
import cv2
import numpy as np
from datetime import datetime

# Add mlflow_integration to path
sys.path.append(str(Path().absolute().parent))

# Import our custom MLflow configuration
from mlflow_integration.mlflow_config import MLflowConfig

print("\n🔧 CONFIGURING MLFLOW")
print("=" * 60)

# Create MLflow configuration
config_mlflow = MLflowConfig(
    tracking_uri="./mlruns",
    experiment_name="RipCatch-Quickstart",
    artifact_location="./mlartifacts"
)

# Setup MLflow (creates experiment if it doesn't exist)
mlflow_setup = config_mlflow.setup_mlflow()

print(f"✅ MLflow Tracking URI: {mlflow_setup['tracking_uri']}")
print(f"✅ Experiment Name: {mlflow_setup['experiment_name']}")
print(f"✅ Experiment ID: {mlflow_setup['experiment_id']}")
print("=" * 60)
print("\n💡 TIP: MLflow is now ready to track your experiments!")
print("   All your training runs will be saved automatically.")

## 🎯 Step 4: Load RipCatch v2.0 Model and Log Basic Info

In [ ]:
# ============================================================================
# STEP 4: Load RipCatch v2.0 Model and Log Basic Info
# ============================================================================

# Define where your model is saved
model_path = "../RipCatch-v2.0/Model/weights/best.pt"

# Start an MLflow run
with mlflow.start_run(run_name="ripcatch_v2.0_baseline") as run:
    
    # Load the YOLO model
    print("📥 Loading RipCatch v2.0 model...")
    model = YOLO(model_path)
    print("✅ Model loaded successfully!")
    
    # Log Model Parameters
    log_param("model_version", "v2.0")
    log_param("model_architecture", "YOLOv8m")
    log_param("model_path", model_path)
    log_param("image_size", config['image_size'])  # Use detected config
    log_param("device", config['device'])  # Use detected device
    log_param("gpu_name", config['gpu_name'])  # Log GPU name
    
    # Log Model Metadata
    log_param("classes", model.names)
    log_param("num_classes", len(model.names))
    
    # Log System Information
    log_param("pytorch_version", torch.__version__)
    log_param("cuda_available", torch.cuda.is_available())
    
    # Print summary
    print("\n" + "="*60)
    print("📊 EXPERIMENT RUN SUMMARY")
    print("="*60)
    print(f"🆔 Run ID: {run.info.run_id}")
    print(f"📦 Model: {model_path}")
    print(f"🏷️  Classes: {model.names}")
    print(f"💻 Device: {config['device']}")
    if config['device'] == 'cuda':
        print(f"🎮 GPU: {config['gpu_name']}")
    print("="*60)
    print("\n✅ All information logged to MLflow!")
    print("💡 TIP: View this run in MLflow UI at http://localhost:5000")

## 📊 Step 5: Run Inference and Log Metrics

In [ ]:
# ============================================================================
# STEP 5: Run Model Validation and Log Performance Metrics
# ============================================================================

# Start a new MLflow run for validation
with mlflow.start_run(run_name="ripcatch_v2.0_validation") as run:
    
    print("🔍 Starting model validation on GPU...")
    print(f"Using device: {config['device']}")
    if config['device'] == 'cuda':
        print(f"GPU: {config['gpu_name']}")
    print()
    
    # Define Validation Data Path
    val_data_path = "../RipCatch-v2.0/Datasets/data.yaml"
    
    # Run Validation with detected hardware settings
    results = model.val(
        data=val_data_path,
        imgsz=config['image_size'],  # Use detected optimal image size
        batch=config['batch_size'],  # Use detected optimal batch size
        conf=0.25,  # Confidence threshold
        iou=0.45,  # IOU threshold
        device=config['device']  # Use detected device (GPU or CPU)
    )
    
    # Extract Metrics from Results
    metrics = results.results_dict
    
    # Log Performance Metrics to MLflow
    log_metric("mAP50", metrics.get('metrics/mAP50(B)', 0))
    log_metric("mAP50-95", metrics.get('metrics/mAP50-95(B)', 0))
    log_metric("precision", metrics.get('metrics/precision(B)', 0))
    log_metric("recall", metrics.get('metrics/recall(B)', 0))
    
    # Log Inference Parameters
    log_param("confidence_threshold", 0.25)
    log_param("iou_threshold", 0.45)
    log_param("batch_size", config['batch_size'])
    log_param("validation_device", config['device'])
    
    # Print Results
    print("\n" + "="*60)
    print("📊 VALIDATION RESULTS")
    print("="*60)
    print(f"✅ Validation Complete!")
    print(f"\n📈 Performance Metrics:")
    print(f"   • mAP@50:      {metrics.get('metrics/mAP50(B)', 0):.4f}")
    print(f"   • mAP@50-95:   {metrics.get('metrics/mAP50-95(B)', 0):.4f}")
    print(f"   • Precision:   {metrics.get('metrics/precision(B)', 0):.4f}")
    print(f"   • Recall:      {metrics.get('metrics/recall(B)', 0):.4f}")
    print("="*60)
    print("\n💡 WHAT DO THESE NUMBERS MEAN?")
    print("   • Higher numbers are better (max is 1.0 = 100%)")
    print("   • mAP@50 > 0.85 = Excellent model!")
    print("   • Precision > 0.85 = Very reliable detections")
    print("   • Recall > 0.85 = Finds most rip currents")
    print("\n✅ All metrics logged to MLflow!")
    print("="*60)

## 🖼️ Step 6: Log Artifacts (Images, Plots, Model)

In [ ]:
# ============================================================================
# STEP 5: Log Artifacts (Files, Plots, Models)
# ============================================================================
# This cell creates and saves files to MLflow for later reference

# WHAT ARE ARTIFACTS?
# ------------------
# Artifacts are files you want to save with your experiment:
# - Model weights (.pt files)
# - Plots and graphs (.png, .jpg)
# - Text files with notes (.txt)
# - Any other files you might need later

# Think of artifacts as attachments to your experiment notebook!

import matplotlib.pyplot as plt  # For creating plots
import tempfile  # For creating temporary folders
import shutil  # For deleting temporary folders

# Start a new MLflow run for logging artifacts
with mlflow.start_run(run_name="ripcatch_v2.0_artifacts") as run:
    
    print("📦 Starting to log artifacts...")
    print("This will save model files, plots, and other data.\n")
    
    # STEP 5.1: Create Temporary Directory
    # ------------------------------------
    # We need a temporary folder to store files before uploading to MLflow
    # Think of it as a staging area before filing documents
    temp_dir = tempfile.mkdtemp()
    print(f"📁 Created temporary directory: {temp_dir}\n")
    
    # ===========================================================================
    # ARTIFACT 1: Log Model Weights File
    # ===========================================================================
    print("1️⃣  Logging model weights...")
    # This saves your trained model file (.pt) to MLflow
    # WHY? So you can download and use this exact model later!
    mlflow.log_artifact(model_path, "model")  # Save to "model" folder in MLflow
    print("   ✅ Model weights logged (best.pt)\n")
    
    # ===========================================================================
    # ARTIFACT 2: Create and Log Performance Visualization
    # ===========================================================================
    print("2️⃣  Creating performance visualization...")
    
    # Create a bar chart showing model performance
    fig, ax = plt.subplots(figsize=(10, 6))  # Create a 10x6 inch plot
    
    # Define metric names and values
    metrics_names = ['mAP@50', 'mAP@50-95', 'Precision', 'Recall']
    metrics_values = [0.8864, 0.6532, 0.8903, 0.8951]  # RipCatch v2.0 results
    
    # Create colorful bars
    # COLORS: green, blue, orange, red (to make it visually appealing)
    bars = ax.bar(metrics_names, metrics_values, 
                  color=['#2ecc71', '#3498db', '#f39c12', '#e74c3c'])
    
    # Customize the plot
    ax.set_ylim(0, 1.0)  # Y-axis from 0 to 1 (0% to 100%)
    ax.set_ylabel('Score', fontsize=12)  # Label for Y-axis
    ax.set_title('RipCatch v2.0 Performance Metrics', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)  # Add subtle grid lines
    
    # Add value labels on top of each bar
    # This makes it easy to see exact numbers
    for bar in bars:
        height = bar.get_height()  # Get bar height (metric value)
        ax.text(bar.get_x() + bar.get_width()/2.,  # X position (center of bar)
                height,  # Y position (top of bar)
                f'{height:.4f}',  # Text to display (4 decimal places)
                ha='center',  # Horizontal alignment: center
                va='bottom')  # Vertical alignment: bottom
    
    # Save plot to temporary directory
    plot_path = f"{temp_dir}/performance_metrics.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')  # High quality (150 DPI)
    plt.close()  # Close to free up memory
    
    # Log plot to MLflow
    mlflow.log_artifact(plot_path, "plots")  # Save to "plots" folder
    print("   ✅ Performance plot saved and logged\n")
    
    # ===========================================================================
    # ARTIFACT 3: Create and Log Model Information Text File
    # ===========================================================================
    print("3️⃣  Creating model information file...")
    
    # Create a text file with model details
    # This is like writing a summary card for your model
    info_path = f"{temp_dir}/model_info.txt"
    
    with open(info_path, 'w') as f:
        # Write header
        f.write("RipCatch v2.0 Model Information\n")
        f.write("="*50 + "\n\n")
        
        # Write model architecture details
        f.write("🏗️  ARCHITECTURE:\n")
        f.write(f"   • Model Type: YOLOv8m (Medium)\n")
        f.write(f"   • Framework: Ultralytics YOLOv8\n")
        f.write(f"   • Classes: {model.names}\n")
        f.write(f"   • Input Size: 640x640 pixels\n")
        f.write(f"   • Parameters: ~26 Million\n")
        f.write(f"   • Model Size: ~52 MB\n\n")
        
        # Write performance metrics
        f.write("📊 PERFORMANCE METRICS:\n")
        f.write(f"   • mAP@50:      88.64% (Excellent!)\n")
        f.write(f"   • mAP@50-95:   65.32% (Very Good)\n")
        f.write(f"   • Precision:   89.03% (Reliable)\n")
        f.write(f"   • Recall:      89.51% (Comprehensive)\n\n")
        
        # Write usage notes
        f.write("💡 USAGE NOTES:\n")
        f.write(f"   • Best for: Beach surveillance and rip current detection\n")
        f.write(f"   • Confidence threshold: 0.25 (25%)\n")
        f.write(f"   • Recommended IOU: 0.45 (45%)\n")
        f.write(f"   • Inference speed: ~30-40 FPS on GPU\n")
    
    # Log text file to MLflow
    mlflow.log_artifact(info_path, "info")  # Save to "info" folder
    print("   ✅ Model info file created and logged\n")
    
    # ===========================================================================
    # CLEANUP: Delete Temporary Directory
    # ===========================================================================
    # Remove temporary files (they're now saved in MLflow)
    shutil.rmtree(temp_dir)
    
    # Print summary
    print("="*60)
    print("✅ ALL ARTIFACTS SUCCESSFULLY LOGGED!")
    print("="*60)
    print(f"🆔 Run ID: {run.info.run_id}")
    print(f"\n📦 Saved Artifacts:")
    print(f"   1. Model weights (best.pt)")
    print(f"   2. Performance visualization (performance_metrics.png)")
    print(f"   3. Model information (model_info.txt)")
    print("\n💡 HOW TO VIEW THESE FILES:")
    print(f"   1. Open MLflow UI: http://localhost:5000")
    print(f"   2. Click on this run: ripcatch_v2.0_artifacts")
    print(f"   3. Scroll down to 'Artifacts' section")
    print(f"   4. Download any file you need!")
    print("="*60)

## 🌐 Step 7: Launch MLflow UI

You can view all logged experiments in the MLflow UI.

In [ ]:
# Launch MLflow UI
# Run this in a separate terminal:
# mlflow ui --backend-store-uri ./mlruns --port 5000

print("🌐 To view the MLflow UI:")
print("   1. Open a new terminal")
print("   2. Run: mlflow ui --backend-store-uri ./mlruns --port 5000")
print("   3. Open http://localhost:5000 in your browser")
print("\n📊 You'll see:")
print("   - All experiment runs")
print("   - Parameters and metrics for each run")
print("   - Logged artifacts (models, plots, files)")
print("   - Comparison charts and visualizations")

## 🎓 Step 8: Query Experiments Programmatically

In [ ]:
# ============================================================================
# STEP 7: Query Experiments Programmatically
# ============================================================================
# This cell shows you how to access your MLflow data using Python code

# WHAT IS "PROGRAMMATIC" ACCESS?
# ------------------------------
# Instead of clicking through the MLflow UI website, you can use Python
# code to automatically search, compare, and analyze your experiments.
# Think of it like using code to search through your experiment notebook!

from mlflow.tracking import MlflowClient  # Import MLflow's Python API

# STEP 7.1: Create MLflow Client
# -------------------------------
# The client is your interface to talk to MLflow
# It's like a librarian who helps you find books (experiments) in a library
client = MlflowClient()

# STEP 7.2: Get Experiment by Name
# --------------------------------
# Find the experiment we created earlier
experiment = client.get_experiment_by_name("RipCatch-Quickstart")

print("="*60)
print("📁 EXPERIMENT INFORMATION")
print("="*60)
print(f"Name: {experiment.name}")
print(f"ID: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"Lifecycle Stage: {experiment.lifecycle_stage}")
print("="*60 + "\n")

# STEP 7.3: Search for All Runs in This Experiment
# ------------------------------------------------
# Get all the runs (training sessions) in this experiment
# PARAMETERS:
# - experiment_ids: Which experiment to search (can search multiple)
# - order_by: How to sort results (newest first)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],  # Search in our experiment
    order_by=["start_time DESC"]  # Sort by start time (newest first)
)

print("="*60)
print(f"📊 EXPERIMENT RUNS SUMMARY")
print("="*60)
print(f"Total Runs Found: {len(runs)}")
print(f"Showing: First {min(5, len(runs))} runs\n")

# STEP 7.4: Display Information for Each Run
# ------------------------------------------
# Loop through the first 5 runs and show their details
for i, run in enumerate(runs[:5], 1):  # [:5] means first 5 runs
    
    print(f"{'='*60}")
    print(f"RUN #{i}: {run.info.run_name}")
    print(f"{'='*60}")
    
    # Basic run information
    print(f"🆔 Run ID: {run.info.run_id}")
    print(f"📅 Status: {run.info.status}")
    
    # Convert timestamp to readable date
    start_time = datetime.fromtimestamp(run.info.start_time/1000)
    print(f"🕐 Start Time: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
    
    # Show parameters (settings used)
    if run.data.params:
        print(f"\n⚙️  Parameters ({len(run.data.params)} total):")
        # Show first 3 parameters
        for key, value in list(run.data.params.items())[:3]:
            print(f"   • {key}: {value}")
        if len(run.data.params) > 3:
            print(f"   ... and {len(run.data.params) - 3} more")
    
    # Show metrics (performance measurements)
    if run.data.metrics:
        print(f"\n📊 Metrics ({len(run.data.metrics)} total):")
        # Show first 3 metrics
        for key, value in list(run.data.metrics.items())[:3]:
            print(f"   • {key}: {value:.4f}")
        if len(run.data.metrics) > 3:
            print(f"   ... and {len(run.data.metrics) - 3} more")
    
    # Show tags (labels)
    if run.data.tags:
        print(f"\n🏷️  Tags ({len(run.data.tags)} total):")
        # Show first 2 tags
        for key, value in list(run.data.tags.items())[:2]:
            print(f"   • {key}: {value}")
    
    print()  # Empty line for spacing

print("="*60)
print("✅ QUERY COMPLETE!")
print("="*60)
print("\n💡 WHAT YOU CAN DO WITH THIS:")
print("   1. Compare different runs programmatically")
print("   2. Find the best performing model automatically")
print("   3. Export data to CSV or Excel for analysis")
print("   4. Create custom visualizations")
print("   5. Build automated ML pipelines")
print("\n💡 NEXT STEPS:")
print("   • Try modifying the search query to filter runs")
print("   • Use client.search_runs() with different filters")
print("   • Export metrics to a pandas DataFrame for analysis")
print("="*60)

## ✅ Next Steps

Congratulations! You've completed the MLflow quickstart. Here's what you learned:

1. ✅ Install and configure MLflow
2. ✅ Create and manage experiments
3. ✅ Log parameters, metrics, and artifacts
4. ✅ View results in MLflow UI
5. ✅ Query experiments programmatically

### 📚 Continue Learning:

- **02_experiment_tracking.ipynb** - Deep dive into experiment tracking
- **03_model_comparison.ipynb** - Compare RipCatch v1.0 vs v2.0
- **04_hyperparameter_tuning.ipynb** - Optimize model hyperparameters
- **05_production_deployment.ipynb** - Deploy models to production

### 🔗 Useful Resources:

- [MLflow Documentation](https://mlflow.org/docs/latest/index.html)
- [MLflow Python API](https://mlflow.org/docs/latest/python_api/index.html)
- [YOLOv8 Documentation](https://docs.ultralytics.com/)
- [RipCatch GitHub Repository](https://github.com/yourusername/RipCatch)